# RouteFinder — Training on Kaggle

**Before running:**
1. Enable **Internet** (Settings → Internet → On)
2. Enable **GPU** (Settings → Accelerator → GPU T4 x1)
3. Add two secrets (Add-ons → Secrets → Add new secret), then toggle both **on** for this notebook:
   - `HF_TOKEN` — your HuggingFace token
   - `GITHUB_TOKEN` — a GitHub fine-grained PAT with read-only **Contents** permission on this repo

Then click **Run All**. The best checkpoint is saved to `/kaggle/working/checkpoints/` and zipped for download at the end.

In [ ]:
# ── 1. Install dependencies not pre-installed on Kaggle ───────────────────────
!pip install -q lightly pytorch-metric-learning
# timm, pytorch-lightning, datasets, torch are already on Kaggle

In [ ]:
# ── 2. Pull the repo and add it to the Python path ───────────────────────────
import os, sys
from kaggle_secrets import UserSecretsClient

gh_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{gh_token}@github.com/Declan-Bracken/RouteFinder.git"
REPO_DIR = "/kaggle/working/RouteFinder"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Repo ready.")

In [ ]:
# ── 3. Load HF token from Kaggle Secrets ──────────────────────────────────────
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
print("HF token loaded:", hf_token[:8] + "...")

In [ ]:
# ── 4. Configure the run ──────────────────────────────────────────────────────
# Edit anything here. Everything else in train.py has sensible defaults.
from train.train import Config

cfg = Config(
    hf_token          = hf_token,
    hf_dataset        = "DeclanBracken/RouteFinderDatasetV2",

    # Model — start with frozen backbone (num_unfrozen_blocks=0).
    # Increase to 1 or 2 once you have more field data.
    num_unfrozen_blocks = 0,

    # Training
    batch_size        = 128,
    lr                = 1e-3,
    temperature       = 0.07,
    max_epochs        = 100,
    warmup_epochs     = 5,
    patience          = 15,

    # Checkpoints land in /kaggle/working/checkpoints/
    checkpoint_dir    = "/kaggle/working/checkpoints",
)

print(cfg)

In [ ]:
# ── 5. Train ──────────────────────────────────────────────────────────────────
from train.train import train

best_ckpt = train(cfg)
print(f"\nBest checkpoint: {best_ckpt}")

In [ ]:
# ── 6. Quick eval on val split ────────────────────────────────────────────────
# Reloads the best checkpoint and prints final Recall@K.
import torch
from torch.utils.data import DataLoader
from sklearn.neighbors import NearestNeighbors
from datasets import load_dataset
from train.train import RouteFinderModel, EvalDataset, Config
import random

device = "cuda" if torch.cuda.is_available() else "cpu"

ds = load_dataset(cfg.hf_dataset, token=cfg.hf_token)
all_routes = list(set(ds["train"]["label"]))
random.seed(42)
n_test = max(1, int(len(all_routes) * cfg.test_split))
test_routes = set(random.sample(all_routes, n_test))
test_split = ds["train"].filter(lambda x: x["label"] in test_routes)

model = RouteFinderModel.load_from_checkpoint(best_ckpt).to(device).eval()
loader = DataLoader(EvalDataset(test_split), batch_size=64, num_workers=2)

embeddings, labels = [], []
with torch.no_grad():
    for imgs, lbls in loader:
        embeddings.append(model(imgs.to(device)).cpu())
        labels.extend(lbls.tolist())

emb = torch.cat(embeddings).numpy()
knn = NearestNeighbors(n_neighbors=11, metric="cosine")
knn.fit(emb)
_, indices = knn.kneighbors(emb)

print("\n── Final Recall@K ──")
for k in [1, 3, 5, 10]:
    hits = sum(labels[i] in [labels[j] for j in indices[i][1:k+1]] for i in range(len(labels)))
    print(f"  Recall@{k}: {hits/len(labels):.4f}")

In [ ]:
# ── 7. Zip the best checkpoint for download ───────────────────────────────────
import zipfile
from IPython.display import FileLink

zip_path = "/kaggle/working/routefinder_best.zip"
ckpt_name = best_ckpt.split("/")[-1]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(best_ckpt, arcname=ckpt_name)

print(f"Saved: {zip_path}")
FileLink("routefinder_best.zip")